# YOLO Hand-Bill Detection - Colab Training

Trains a `hand_bill` detector on the GPU runtime of Google Colab.

**Steps:**
1. **Runtime > Change runtime type > T4 GPU**
2. Run cells in order.
3. When asked, upload `bill_hand_dataset_colab.zip`.
4. After training, download `best.pt` (and `results.png`).

In [ ]:
# 1) Install Ultralytics
!pip install -q ultralytics
from ultralytics import YOLO
print('Ultralytics', YOLO.__module__ and __import__('ultralytics').__version__)

## 2) Upload your dataset zip

Use the local file `bill_hand_dataset/bill_hand_dataset_colab.zip` (already has a relative-path `data.yaml`).

Expected layout inside the zip:
```
data.yaml
yolo/images/{train,val}/...
yolo/labels/{train,val}/...
```

In [ ]:
from google.colab import files
import zipfile, glob, os

uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

!rm -rf /content/dataset
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/dataset')
print('Extracted to /content/dataset')

# sanity checks
tr = glob.glob('/content/dataset/yolo/images/train/*')
va = glob.glob('/content/dataset/yolo/images/val/*')
print(f'train images: {len(tr)} | val images: {len(va)}')
print('data.yaml:')
print(open('/content/dataset/data.yaml').read())

## 3) Training configuration

Edit `EPOCHS`, `IMGSZ`, `BATCH` if needed. On a T4 GPU each epoch on this tiny dataset takes a few seconds.

In [ ]:
EPOCHS = 100
IMGSZ = 640
BATCH = 32
MODEL = 'yolo11n.pt'          # fastest; switch to yolo11s.pt for a bit more accuracy
DATA = '/content/dataset/data.yaml'
RUN_NAME = 'bill_hand_colab'

In [ ]:
# 4) Train
model = YOLO(MODEL)
results = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project='/content/runs',
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    patience=30,
)

## 5) Results

Wait for the training cell above to finish, then run this.

In [ ]:
import pandas as pd, os
run_dir = f'/content/runs/detect/{RUN_NAME}'
df = pd.read_csv(f'{run_dir}/results.csv')
print('Last 5 rows:')
print(df.tail())
print()
best = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f"Best epoch: {int(best['epoch'])}  mAP50={best['metrics/mAP50(B)']:.4f}  mAP50-95={best['metrics/mAP50-95(B)']:.4f}")
print(f"Final:        mAP50={df['metrics/mAP50(B)'].iloc[-1]:.4f}  mAP50-95={df['metrics/mAP50-95(B)'].iloc[-1]:.4f}")
print()
print('Files in run dir:', sorted(os.listdir(run_dir)))

In [ ]:
# 6) Download the trained weights + curves
from google.colab import files
run_dir = f'/content/runs/detect/{RUN_NAME}'
files.download(f'{run_dir}/weights/best.pt')
try:
    files.download(f'{run_dir}/results.png')
except Exception as e:
    print('results.png not found:', e)

## 7) Quick visual test on validation images
```python
from ultralytics import YOLO
best = YOLO('/content/runs/detect/bill_hand_colab/weights/best.pt')
best.predict('/content/dataset/yolo/images/val', conf=0.25, save=True, project='/content/val_test')
```